# 1. Basic graph classes

This notebook contains the `Node` and `Graph` classes from the initial
implementation. They represent **simple, undirected graphs**: each edge joins
two distinct vertices, and there is at most one edge between a pair of vertices.

`02_algorithms.ipynb` loads these definitions using `%run 01_graphs.ipynb`.
This notebook only defines classes; running it does not generate graphs,
execute experiments, or save results. Use the project's `.venv` kernel.
The other notebooks locate these definitions from either the project root
or the `notebooks/` directory.

In [ ]:
import uuid

## A vertex: identity, neighbours, and degree

A `Node` has a UUID identifier, a set of neighbouring nodes, and a cached degree.
Equality and hashing use the UUID, which also lets copied graphs retain the
identities of the original vertices. UUIDs are labels; they do not affect the
probabilistic model.

`join()` and `unjoin()` update both endpoints. They are the original low-level
operations; when a node belongs to a `Graph`, use `Graph.add_edge()` and
`Graph.remove_edge()` so the graph's edge set stays in sync as well. The printed
messages for invalid or repeated operations are retained from the initial
notebook. Do not edit `neighbours`, `degree`, or `id` directly in experiments.

In [ ]:
class Node:
    """A vertex with a UUID label and mutable undirected adjacency.

    Neighbours are stored once in a set; ``degree`` caches that set's size.
    Joining or unjoining a pair updates both vertices together.
    """

    def __init__(self):
        self.id = uuid.uuid4()
        self.neighbours = set()
        self.degree = 0

    def join(self, node):
        """Join two distinct vertices, updating both cached degrees."""
        if node is self:
            print("Node cannot be joined to itself")
        elif node not in self.neighbours:
            self.neighbours.add(node)
            node.neighbours.add(self)
            self.degree += 1
            node.degree += 1
        else:
            print("Nodes are already joined")

    def unjoin(self, node):
        """Remove an existing adjacency from both endpoints."""
        if node in self.neighbours:
            self.neighbours.remove(node)
            node.neighbours.remove(self)
            self.degree -= 1
            node.degree -= 1
        else:
            print("Nodes are not joined")

    def __eq__(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)

## A graph: vertices and unordered edges

`Graph(n)` starts with $n$ isolated nodes. `nodes` is a set of `Node` objects;
`edges` is a set of two-element `frozenset`s, so an edge has no orientation.
Removing a node also removes all its incident edges.

The original custom `__deepcopy__()` rebuilds nodes and edges with independent
mutable state while preserving UUIDs. Core peeling therefore returns a separate
graph whose surviving node identities can be compared with the input graph.

Two small fields retain the sampling history of a random intersection graph:

- `memberships`: the accepted number of community memberships for each UUID,
  including zero counts;
- `communities`: an immutable tuple of communities, each a tuple of UUID slots.
  A UUID may occur more than once in a community.

These fields describe the **original sampled memberships**, not the current
adjacency. Removing vertices or taking a core keeps this provenance intact;
use `nodes`, `edges`, and each node's `degree` for the current graph. Manually
constructed graphs start with empty membership metadata.

In [ ]:
class Graph:
    """The original mutable simple graph, plus original membership metadata."""

    def __init__(self, n=0):
        self.nodes = {Node() for _ in range(n)}
        self.edges = set()
        self.memberships = {}
        self.communities = ()

    def add_node(self, node):
        """Add a vertex; its existing identity and adjacency are unchanged."""
        if node not in self.nodes:
            self.nodes.add(node)
        else:
            print("Node already in graph")

    def remove_node(self, node):
        """Remove a vertex and its incident edges from this graph."""
        if node in self.nodes:
            # Copy the neighbour set because remove_edge mutates it.
            for neighbour in node.neighbours.copy():
                self.remove_edge(node, neighbour)
            self.nodes.remove(node)
        else:
            print("Node not in graph")

    def add_edge(self, node1, node2):
        """Add an edge between two vertices already belonging to the graph."""
        if node1 not in self.nodes or node2 not in self.nodes:
            print("Nodes not in graph")
        elif node1 is node2:
            print("Node cannot be joined to itself")
        elif frozenset({node1, node2}) in self.edges:
            print("Edge already in graph")
        else:
            node1.join(node2)
            self.edges.add(frozenset({node1, node2}))

    def remove_edge(self, node1, node2):
        """Remove an edge and update both endpoints' neighbour sets."""
        edge = frozenset({node1, node2})
        if edge in self.edges:
            node1.unjoin(node2)
            self.edges.remove(edge)
        else:
            print("Edge not in graph")

    def __deepcopy__(self, memo):
        """Copy graph state independently while retaining vertex UUID labels."""
        new_graph = Graph()
        memo[id(self)] = new_graph

        # Map original vertices to new vertices with the same UUIDs.
        node_map = {}
        for node in self.nodes:
            new_node = Node()
            new_node.id = node.id
            node_map[node] = new_node
            memo[id(node)] = new_node
            new_graph.add_node(new_node)

        # Reconstruct edges rather than sharing mutable adjacency sets.
        for edge in self.edges:
            node1, node2 = tuple(edge)
            new_graph.add_edge(node_map[node1], node_map[node2])

        # Counts are integers, and community slots are immutable UUID tuples.
        # Keep the original sampling history even if the copy is later pruned.
        new_graph.memberships = self.memberships.copy()
        new_graph.communities = self.communities
        return new_graph